In [1]:
# Transformation Results Visualization Dashboard
# ==============================================
# This notebook provides interactive visualizations for your data pipeline results
# Run each cell to see how your data transformations performed

import sys
import subprocess
import os
import json
import pickle
import warnings
warnings.filterwarnings('ignore')

print("🚀 Data Pipeline Visualization Dashboard")
print("="*50)

# Install required packages
def install_packages():
    """Install required packages for visualization"""
    required_packages = [
        'pandas>=1.0.0',
        'plotly>=5.0.0',
        'numpy'
    ]
    
    print("📦 Installing required packages...")
    for package in required_packages:
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", package], 
                                stdout=subprocess.DEVNULL, 
                                stderr=subprocess.DEVNULL)
        except subprocess.CalledProcessError as e:
            print(f"⚠️ Warning: Could not install {package}: {e}")
    
    print("✅ Package installation complete!")

# Install packages first
install_packages()

🚀 Data Pipeline Visualization Dashboard
📦 Installing required packages...
✅ Package installation complete!


In [2]:
# Now import the packages
try:
    import pandas as pd
    import plotly.graph_objects as go
    import plotly.express as px
    from plotly.subplots import make_subplots
    import numpy as np
    PLOTLY_AVAILABLE = True
    print("✅ All visualization libraries imported successfully!")
except ImportError as e:
    print(f"⚠️ Warning: Some visualization libraries not available: {e}")
    print("📊 Will use text-based summaries instead of interactive plots")
    import pandas as pd
    import numpy as np
    PLOTLY_AVAILABLE = False

# Configuration
PIPELINE_OUTPUT_PATH = './pipeline_output'
TRANSFORMATIONS = {
    'deduplication': {
        'folder': 'deduplicated',
        'stats_file': 'deduplication_stats.json',
        'plotly_file': 'deduplication_plotly.json',
        'data_file': 'deduplicated_data.pkl',
        'title': '🧹 Duplicate Removal',
        'color': '#3498db'
    },
    'outlier_removal': {
        'folder': 'outliers_removed', 
        'stats_file': 'outlier_removal_stats.json',
        'plotly_file': 'outlier_removal_plotly.json',
        'data_file': 'outliers_removed_data.pkl',
        'title': '🎯 Outlier Detection',
        'color': '#e74c3c'
    },
    'null_cleaning': {
        'folder': 'null_cleaned',
        'stats_file': 'null_cleaning_stats.json', 
        'plotly_file': 'null_cleaning_plotly.json',
        'data_file': 'nulls_cleaned_data.pkl',
        'title': '🧽 Null Value Cleaning',
        'color': '#2ecc71'
    }
}

# Helper Functions
def load_json_safe(file_path):
    """Safely load JSON file"""
    try:
        if os.path.exists(file_path):
            with open(file_path, 'r') as f:
                return json.load(f)
    except Exception as e:
        print(f"⚠️ Warning: Could not load {file_path}: {e}")
    return None

def load_pickle_safe(file_path):
    """Safely load pickle file"""
    try:
        if os.path.exists(file_path):
            with open(file_path, 'rb') as f:
                return pickle.load(f)
    except Exception as e:
        print(f"⚠️ Warning: Could not load {file_path}: {e}")
    return None

def detect_available_transformations():
    """Detect which transformations have been run"""
    available = {}
    
    for trans_name, config in TRANSFORMATIONS.items():
        folder_path = os.path.join(PIPELINE_OUTPUT_PATH, config['folder'])
        stats_path = os.path.join(folder_path, config['stats_file'])
        
        if os.path.exists(stats_path):
            stats = load_json_safe(stats_path)
            if stats:
                available[trans_name] = {
                    'config': config,
                    'stats': stats,
                    'folder_path': folder_path
                }
                print(f"✅ Found: {config['title']}")
            else:
                print(f"❌ Invalid stats file: {config['title']}")
        else:
            print(f"⚪ Not run: {config['title']}")
    
    return available

def print_ascii_bar_chart(data, title, max_width=50):
    """Print a simple ASCII bar chart"""
    print(f"\n📊 {title}")
    print("-" * (max_width + 20))
    
    if not data:
        print("No data to display")
        return
    
    max_val = max(data.values()) if data.values() else 1
    
    for label, value in data.items():
        bar_length = int((value / max_val) * max_width) if max_val > 0 else 0
        bar = "█" * bar_length
        print(f"{label:<15} │{bar:<{max_width}} │ {value:,}")
    
    print("-" * (max_width + 20))

# Load raw data info
raw_data_path = os.path.join(PIPELINE_OUTPUT_PATH, 'raw_data.pkl')
raw_data = load_pickle_safe(raw_data_path)
raw_data_shape = raw_data.shape if raw_data is not None else (0, 0)

print(f"\n📊 Raw Data: {raw_data_shape[0]:,} rows × {raw_data_shape[1]} columns")

# Detect available transformations  
available_transformations = detect_available_transformations()

if not available_transformations:
    print("\n❌ No transformation results found!")
    print("Please run some transformations first:")
    print("  export TEST_MODE=local")
    print("  python load_data_local.py")
    print("  python remove_duplicates.py")
    print("  python remove_outliers.py") 
    print("  python clean_null_values.py")
else:
    print(f"\n🎉 Found {len(available_transformations)} completed transformations!")


✅ All visualization libraries imported successfully!

📊 Raw Data: 2,918 rows × 42 columns
✅ Found: 🧹 Duplicate Removal
✅ Found: 🎯 Outlier Detection
✅ Found: 🧽 Null Value Cleaning

🎉 Found 3 completed transformations!


In [3]:
# Cell 3: Auto-discovery function
def discover_transformations(base_path='./pipeline_output'):
    """Automatically discover all transformations in pipeline_output folder"""
    transformations = []
    
    print(f"🔍 Scanning {base_path} for transformations...")
    
    if not os.path.exists(base_path):
        print(f"❌ Directory {base_path} not found!")
        return transformations
    
    # Scan each subdirectory
    for item in os.listdir(base_path):
        item_path = os.path.join(base_path, item)
        
        if os.path.isdir(item_path):
            print(f"  📁 Checking {item}/")
            
            # Look for JSON stats files (any *stats.json)
            stats_files = [f for f in os.listdir(item_path) if f.endswith('_stats.json')]
            plotly_files = [f for f in os.listdir(item_path) if f.endswith('_plotly.json')]
            data_files = [f for f in os.listdir(item_path) if f.endswith('.pkl')]
            
            if stats_files and plotly_files:
                # Try to load stats to get transformation info
                stats_file = stats_files[0]
                plotly_file = plotly_files[0]
                data_file = data_files[0] if data_files else None
                
                try:
                    with open(os.path.join(item_path, stats_file), 'r') as f:
                        stats = json.load(f)
                    
                    transformation_type = stats.get('transformation', 'unknown')
                    
                    # Get nice name and icon
                    name_map = {
                        'remove_duplicates': {'name': 'Duplicate Removal', 'icon': '🧹'},
                        'clean_null_values': {'name': 'Null Value Cleaning', 'icon': '🧽'},
                        'remove_outliers': {'name': 'Outlier Removal', 'icon': '🎯'}
                    }
                    
                    info = name_map.get(transformation_type, {'name': transformation_type.title(), 'icon': '📊'})
                    
                    transformations.append({
                        'folder': item,
                        'path': item_path,
                        'type': transformation_type,
                        'name': info['name'],
                        'icon': info['icon'],
                        'stats_file': os.path.join(item_path, stats_file),
                        'plotly_file': os.path.join(item_path, plotly_file),
                        'data_file': os.path.join(item_path, data_file) if data_file else None
                    })
                    
                    print(f"    ✅ Found {info['icon']} {info['name']}")
                    
                except Exception as e:
                    print(f"    ⚠️  Error reading {stats_file}: {e}")
            else:
                print(f"    ❌ No transformation files found")
    
    print(f"\n🎉 Discovered {len(transformations)} transformation(s)!")
    return transformations

# Discover all transformations
transformations = discover_transformations()

🔍 Scanning ./pipeline_output for transformations...
  📁 Checking .ipynb_checkpoints/
    ❌ No transformation files found
  📁 Checking deduplicated/
    ✅ Found 🧹 Duplicate Removal
  📁 Checking null_cleaned/
    ✅ Found 🧽 Null Value Cleaning
  📁 Checking outliers_removed/
    ✅ Found 🎯 Outlier Removal

🎉 Discovered 3 transformation(s)!


In [4]:
# =============================================================================
# SECTION 1: PIPELINE OVERVIEW
# =============================================================================

def create_pipeline_overview():
    """Create pipeline overview visualization"""
    if not available_transformations:
        return None
    
    # Collect data for pipeline flow
    stages = ['Raw Data']
    row_counts = [raw_data_shape[0]]
    colors = ['#95a5a6']
    
    # Add each transformation stage
    for trans_name in ['deduplication', 'outlier_removal', 'null_cleaning']:
        if trans_name in available_transformations:
            trans_data = available_transformations[trans_name]
            stages.append(trans_data['config']['title'])
            row_counts.append(trans_data['stats']['final_rows'])
            colors.append(trans_data['config']['color'])
    
    if PLOTLY_AVAILABLE:
        # Create interactive pipeline flow chart
        fig = go.Figure()
        
        fig.add_trace(go.Scatter(
            x=list(range(len(stages))),
            y=row_counts,
            mode='lines+markers+text',
            name='Data Flow',
            line=dict(width=4, color='#34495e'),
            marker=dict(size=15, color=colors, line=dict(width=2, color='white')),
            text=[f"{count:,}" for count in row_counts],
            textposition="top center",
            textfont=dict(size=12, color='black')
        ))
        
        fig.update_layout(
            title={
                'text': '🔄 Data Pipeline Flow',
                'x': 0.5,
                'font': {'size': 20}
            },
            xaxis=dict(
                tickvals=list(range(len(stages))),
                ticktext=stages,
                tickangle=-45
            ),
            yaxis=dict(title='Number of Rows'),
            height=400,
            showlegend=False,
            hovermode='x unified'
        )
        
        fig.show()
    else:
        # Create text-based pipeline flow
        print("\n🔄 DATA PIPELINE FLOW")
        print("=" * 60)
        for i, (stage, count) in enumerate(zip(stages, row_counts)):
            arrow = " ➜ " if i < len(stages) - 1 else ""
            print(f"{stage}: {count:,} rows{arrow}")
        print("=" * 60)
    
    return stages, row_counts

# Create and show pipeline overview
pipeline_data = create_pipeline_overview()


In [5]:
# =============================================================================
# SECTION 2: TRANSFORMATION STATISTICS TABLE
# =============================================================================

def create_stats_table():
    """Create comprehensive statistics table"""
    if not available_transformations:
        return None
    
    table_data = []
    
    # Add raw data row
    table_data.append({
        'Stage': '🗃️ Raw Data',
        'Rows': f"{raw_data_shape[0]:,}",
        'Columns': f"{raw_data_shape[1]:,}",
        'Change': '-',
        'Details': 'Original dataset'
    })
    
    # Add transformation rows
    for trans_name in ['deduplication', 'outlier_removal', 'null_cleaning']:
        if trans_name in available_transformations:
            trans_data = available_transformations[trans_name]
            stats = trans_data['stats']
            config = trans_data['config']
            
            # Calculate change
            initial_rows = stats.get('initial_rows', raw_data_shape[0])
            final_rows = stats.get('final_rows', initial_rows)
            change = final_rows - initial_rows
            change_pct = (change / initial_rows * 100) if initial_rows > 0 else 0
            
            # Create details based on transformation type
            if trans_name == 'deduplication':
                details = f"Removed {stats.get('removed_rows', 0):,} duplicates"
            elif trans_name == 'outlier_removal':
                method = stats.get('method', 'unknown')
                outliers = stats.get('outliers_detected', 0)
                details = f"Method: {method}, Found {outliers:,} outliers"
            elif trans_name == 'null_cleaning':
                strategy = stats.get('strategy', 'unknown')
                nulls_removed = stats.get('nulls_removed', 0)
                details = f"Strategy: {strategy}, Cleaned {nulls_removed:,} nulls"
            
            table_data.append({
                'Stage': config['title'],
                'Rows': f"{final_rows:,}",
                'Columns': f"{stats.get('final_columns', raw_data_shape[1]):,}",
                'Change': f"{change:+,} ({change_pct:+.1f}%)" if change != 0 else "No change",
                'Details': details
            })
    
    df_table = pd.DataFrame(table_data)
    return df_table

# Create and display stats table
stats_table = create_stats_table()
if stats_table is not None:
    print("\n" + "="*100)
    print("📊 TRANSFORMATION SUMMARY")
    print("="*100)
    print(stats_table.to_string(index=False, max_colwidth=40))
    print("="*100)



📊 TRANSFORMATION SUMMARY
                Stage  Rows Columns          Change                                Details
          🗃️ Raw Data 2,918      42               -                       Original dataset
  🧹 Duplicate Removal   200      42       No change                   Removed 0 duplicates
  🎯 Outlier Detection   200      42 -2,718 (-93.1%)      Method: iqr, Found 2,718 outliers
🧽 Null Value Cleaning   200      42       No change Strategy: fill_median, Cleaned 0 nulls


In [6]:
# =============================================================================
# SECTION 3: INDIVIDUAL TRANSFORMATION VISUALIZATIONS
# =============================================================================

def create_transformation_visualizations():
    """Create detailed visualizations for each transformation"""
    
    for trans_name, trans_data in available_transformations.items():
        config = trans_data['config'] 
        stats = trans_data['stats']
        
        print(f"\n{'='*60}")
        print(f"{config['title']} - DETAILED ANALYSIS")
        print(f"{'='*60}")
        
        if trans_name == 'deduplication':
            create_deduplication_viz(stats, config)
        elif trans_name == 'outlier_removal':
            create_outlier_removal_viz(stats, config)
        elif trans_name == 'null_cleaning':
            create_null_cleaning_viz(stats, config)

def create_deduplication_viz(stats, config):
    """Create deduplication specific visualizations"""
    
    # Print key metrics
    print(f"📊 Key Metrics:")
    print(f"  • Initial Rows: {stats['initial_rows']:,}")
    print(f"  • Duplicates Found: {stats['initial_duplicates']:,}")
    print(f"  • Rows Removed: {stats['removed_rows']:,}")
    print(f"  • Final Rows: {stats['final_rows']:,}")
    print(f"  • Removal Rate: {stats['removal_percentage']:.2f}%")
    
    if PLOTLY_AVAILABLE:
        # Create interactive visualization
        fig = make_subplots(
            rows=1, cols=2,
            subplot_titles=('Dataset Size Comparison', 'Data Composition'),
            specs=[[{"type": "bar"}, {"type": "pie"}]]
        )
        
        # Bar chart
        fig.add_trace(
            go.Bar(
                x=['Original', 'After Deduplication'],
                y=[stats['initial_rows'], stats['final_rows']],
                name='Rows',
                marker_color=['#3498db', '#2ecc71'],
                text=[f"{stats['initial_rows']:,}", f"{stats['final_rows']:,}"],
                textposition='outside'
            ),
            row=1, col=1
        )
        
        # Pie chart
        if stats['removed_rows'] > 0:
            pie_values = [stats['final_rows'], stats['removed_rows']]
            pie_labels = ['Unique Rows', 'Duplicates Removed']
            pie_colors = ['#2ecc71', '#e74c3c']
        else:
            pie_values = [stats['final_rows']]
            pie_labels = ['All Unique Rows']  
            pie_colors = ['#2ecc71']
        
        fig.add_trace(
            go.Pie(
                values=pie_values,
                labels=pie_labels,
                marker_colors=pie_colors,
                hole=0.3
            ),
            row=1, col=2
        )
        
        fig.update_layout(
            title_text=f"{config['title']} Results",
            height=400,
            showlegend=True
        )
        
        fig.show()
    else:
        # Text-based visualization
        data_comparison = {
            'Original': stats['initial_rows'],
            'After Cleaning': stats['final_rows'],
            'Duplicates': stats['removed_rows']
        }
        print_ascii_bar_chart(data_comparison, "Dataset Size Comparison")

def create_outlier_removal_viz(stats, config):
    """Create outlier removal specific visualizations"""
    
    # Print key metrics
    print(f"📊 Key Metrics:")
    print(f"  • Method: {stats['method'].upper()}")
    print(f"  • Initial Rows: {stats['initial_rows']:,}")
    print(f"  • Outliers Detected: {stats['outliers_detected']:,}")
    print(f"  • Final Rows: {stats['final_rows']:,}")
    print(f"  • Outlier Rate: {stats['outlier_percentage']:.2f}%")
    
    if PLOTLY_AVAILABLE:
        # Create interactive visualization
        fig = make_subplots(
            rows=1, cols=2,
            subplot_titles=('Outlier Detection Results', 'Data Distribution'),
            specs=[[{"type": "bar"}, {"type": "pie"}]]
        )
        
        # Bar chart
        fig.add_trace(
            go.Bar(
                x=['Original Rows', 'Final Rows', 'Outliers Detected'],
                y=[stats['initial_rows'], stats['final_rows'], stats['outliers_detected']],
                name='Count',
                marker_color=['#3498db', '#2ecc71', '#e74c3c'],
                text=[f"{stats['initial_rows']:,}", f"{stats['final_rows']:,}", f"{stats['outliers_detected']:,}"],
                textposition='outside'
            ),
            row=1, col=1
        )
        
        # Pie chart
        if stats['outliers_detected'] > 0:
            pie_values = [stats['final_rows'], stats['outliers_detected']]
            pie_labels = ['Clean Data', 'Outliers Detected']
            pie_colors = ['#2ecc71', '#e74c3c']
        else:
            pie_values = [stats['initial_rows']]
            pie_labels = ['All Clean']
            pie_colors = ['#2ecc71']
        
        fig.add_trace(
            go.Pie(
                values=pie_values,
                labels=pie_labels,
                marker_colors=pie_colors,
                hole=0.3
            ),
            row=1, col=2
        )
        
        fig.update_layout(
            title_text=f"{config['title']} Results - {stats['method'].upper()} Method",
            height=400,
            showlegend=True
        )
        
        fig.show()
        
        # Column-wise outlier analysis if available
        if 'outlier_details' in stats and stats['outlier_details']:
            outlier_details = stats['outlier_details']
            
            # Skip isolation forest summary for column-wise analysis
            column_data = {k: v for k, v in outlier_details.items() if k != 'isolation_forest'}
            
            if column_data:
                print(f"\n📊 Column-wise Outlier Analysis:")
                col_names = list(column_data.keys())
                outlier_counts = [details.get('outlier_count', 0) for details in column_data.values()]
                outlier_pcts = [details.get('outlier_percentage', 0) for details in column_data.values()]
                
                fig_cols = go.Figure()
                fig_cols.add_trace(go.Bar(
                    x=col_names,
                    y=outlier_counts,
                    name='Outlier Count',
                    marker_color='#e74c3c',
                    text=[f"{count} ({pct:.1f}%)" for count, pct in zip(outlier_counts, outlier_pcts)],
                    textposition='outside'
                ))
                
                fig_cols.update_layout(
                    title='Outliers by Column',
                    xaxis_title='Columns',
                    yaxis_title='Outlier Count',
                    height=400,
                    xaxis_tickangle=-45
                )
                
                fig_cols.show()
    else:
        # Text-based visualization
        data_comparison = {
            'Original Rows': stats['initial_rows'],
            'Clean Rows': stats['final_rows'],
            'Outliers Found': stats['outliers_detected']
        }
        print_ascii_bar_chart(data_comparison, f"Outlier Detection Results ({stats['method'].upper()})")
        
        # Column-wise analysis (text)
        if 'outlier_details' in stats and stats['outlier_details']:
            column_data = {k: v for k, v in stats['outlier_details'].items() if k != 'isolation_forest'}
            if column_data:
                print(f"\n📊 Outliers by Column:")
                for col, details in column_data.items():
                    count = details.get('outlier_count', 0)
                    pct = details.get('outlier_percentage', 0)
                    print(f"  • {col}: {count:,} outliers ({pct:.1f}%)")

def create_null_cleaning_viz(stats, config):
    """Create null cleaning specific visualizations"""
    
    # Print key metrics
    print(f"📊 Key Metrics:")
    print(f"  • Strategy: {stats['strategy'].upper()}")
    print(f"  • Initial Shape: {stats['initial_rows']:,} rows × {stats['initial_columns']} columns")
    print(f"  • Final Shape: {stats['final_rows']:,} rows × {stats['final_columns']} columns")
    print(f"  • Initial Nulls: {stats['initial_nulls']:,}")
    print(f"  • Final Nulls: {stats['final_nulls']:,}")
    print(f"  • Nulls Cleaned: {stats['nulls_removed']:,}")
    
    if PLOTLY_AVAILABLE:
        # Create comprehensive visualization
        fig = make_subplots(
            rows=2, cols=2,
            subplot_titles=(
                'Before/After Comparison',
                'Null Value Progress', 
                'Missing Values by Column (Top 10 Before)',
                'Missing Values by Column (Top 10 After)'
            ),
            specs=[
                [{"type": "bar"}, {"type": "pie"}],
                [{"type": "bar"}, {"type": "bar"}]
            ]
        )
        
        # Before/After comparison
        categories = ['Rows', 'Columns', 'Total Nulls']
        before_values = [stats['initial_rows'], stats['initial_columns'], stats['initial_nulls']]
        after_values = [stats['final_rows'], stats['final_columns'], stats['final_nulls']]
        
        fig.add_trace(
            go.Bar(x=categories, y=before_values, name='Before', marker_color='#3498db'),
            row=1, col=1
        )
        fig.add_trace(
            go.Bar(x=categories, y=after_values, name='After', marker_color='#2ecc71'),
            row=1, col=1
        )
        
        # Pie chart for null progress
        if stats['nulls_removed'] > 0:
            pie_values = [stats['final_nulls'], stats['nulls_removed']]
            pie_labels = ['Remaining Nulls', 'Nulls Cleaned']
            pie_colors = ['#f39c12', '#2ecc71']
        else:
            pie_values = [stats['initial_nulls']] if stats['initial_nulls'] > 0 else [1]
            pie_labels = ['All Clean'] if stats['initial_nulls'] == 0 else ['No Change']
            pie_colors = ['#2ecc71']
        
        fig.add_trace(
            go.Pie(values=pie_values, labels=pie_labels, marker_colors=pie_colors, hole=0.3),
            row=1, col=2
        )
        
        # Column-wise missing values analysis
        if 'missing_before' in stats and stats['missing_before']:
            missing_before = stats['missing_before']
            missing_after = stats.get('missing_after', {})
            
            # Get top columns with missing values
            sorted_cols = sorted(missing_before.items(), 
                               key=lambda x: x[1]['missing_percentage'], 
                               reverse=True)[:10]
            
            col_names = [col[0] for col in sorted_cols]
            missing_pcts_before = [col[1]['missing_percentage'] for col in sorted_cols]
            missing_pcts_after = [missing_after.get(col, {}).get('missing_percentage', 0) 
                                for col in col_names]
            
            # Before - column missing percentages
            fig.add_trace(
                go.Bar(
                    x=col_names, 
                    y=missing_pcts_before, 
                    name='Before', 
                    marker_color='#e74c3c',
                    text=[f"{pct:.1f}%" for pct in missing_pcts_before],
                    textposition='outside'
                ),
                row=2, col=1
            )
            
            # After - column missing percentages  
            fig.add_trace(
                go.Bar(
                    x=col_names, 
                    y=missing_pcts_after, 
                    name='After', 
                    marker_color='#2ecc71',
                    text=[f"{pct:.1f}%" for pct in missing_pcts_after],
                    textposition='outside'
                ),
                row=2, col=2
            )
        
        fig.update_layout(
            title_text=f"{config['title']} Results - {stats['strategy'].upper()} Strategy",
            height=800,
            showlegend=True
        )
        
        fig.update_xaxes(tickangle=-45, row=2, col=1)
        fig.update_xaxes(tickangle=-45, row=2, col=2)
        fig.update_yaxes(title_text="Missing %", row=2, col=1)
        fig.update_yaxes(title_text="Missing %", row=2, col=2)
        
        fig.show()
    else:
        # Text-based visualization
        data_comparison = {
            'Initial Rows': stats['initial_rows'],
            'Final Rows': stats['final_rows'],
            'Initial Nulls': stats['initial_nulls'],
            'Final Nulls': stats['final_nulls']
        }
        print_ascii_bar_chart(data_comparison, f"Null Cleaning Results ({stats['strategy'].upper()})")
        
        # Column-wise analysis (text)
        if 'missing_before' in stats and stats['missing_before']:
            print(f"\n📊 Top Columns with Missing Values (Before → After):")
            missing_before = stats['missing_before']
            missing_after = stats.get('missing_after', {})
            
            sorted_cols = sorted(missing_before.items(), 
                               key=lambda x: x[1]['missing_percentage'], 
                               reverse=True)[:10]
            
            for col, before_info in sorted_cols:
                before_pct = before_info['missing_percentage']
                after_pct = missing_after.get(col, {}).get('missing_percentage', 0)
                print(f"  • {col}: {before_pct:.1f}% → {after_pct:.1f}%")

# Create all transformation visualizations
create_transformation_visualizations()


🧹 Duplicate Removal - DETAILED ANALYSIS
📊 Key Metrics:
  • Initial Rows: 200
  • Duplicates Found: 0
  • Rows Removed: 0
  • Final Rows: 200
  • Removal Rate: 0.00%



🎯 Outlier Detection - DETAILED ANALYSIS
📊 Key Metrics:
  • Method: IQR
  • Initial Rows: 2,918
  • Outliers Detected: 2,718
  • Final Rows: 200
  • Outlier Rate: 93.15%



📊 Column-wise Outlier Analysis:



🧽 Null Value Cleaning - DETAILED ANALYSIS
📊 Key Metrics:
  • Strategy: FILL_MEDIAN
  • Initial Shape: 200 rows × 42 columns
  • Final Shape: 200 rows × 42 columns
  • Initial Nulls: 224
  • Final Nulls: 224
  • Nulls Cleaned: 0


In [7]:
# =============================================================================
# SECTION 5: RENDER ALL CHARTS FROM TRANSFORMATION DATA
# =============================================================================

def render_chart(chart_name, chart_data, transformation_name):
    """Render a single chart from plotly data"""
    try:
        if not PLOTLY_AVAILABLE:
            print(f"  📊 {chart_name}: Chart data available but plotly not installed")
            return False
        
        if 'data' not in chart_data or 'layout' not in chart_data:
            print(f"  ⚠️ {chart_name}: Invalid chart structure")
            return False
        
        # Create figure from saved plotly data
        fig = go.Figure(data=chart_data['data'], layout=chart_data['layout'])
        
        # Update layout for better notebook display
        fig.update_layout(
            title_font_size=16,
            height=400,
            showlegend=True,
            template="plotly_white"
        )
        
        fig.show()
        return True
        
    except Exception as e:
        print(f"  ❌ {chart_name}: Error rendering chart - {str(e)}")
        return False

def render_all_charts():
    """Render all charts from all transformations"""
    total_charts = 0
    successful_charts = 0
    
    print("\n" + "=" * 70)
    print("📊 RENDERING ALL TRANSFORMATION CHARTS")
    print("=" * 70)
    
    if not available_transformations:
        print("❌ No transformation data available for chart rendering")
        return 0, 0
    
    for trans_name, trans_data in available_transformations.items():
        config = trans_data['config']
        stats = trans_data['stats']
        
        # Load plotly data
        plotly_path = os.path.join(trans_data['folder_path'], config['plotly_file'])
        plotly_data = load_json_safe(plotly_path)
        
        print(f"\n{config['title']} Charts")
        print("-" * 50)
        
        if not plotly_data:
            print(f"❌ No chart data found for {config['title']}")
            continue
        
        chart_count = 0
        for chart_name, chart_data in plotly_data.items():
            if chart_name == 'summary':
                continue
            
            print(f"\n📈 {chart_name.replace('_', ' ').title()}")
            
            if render_chart(chart_name, chart_data, config['title']):
                chart_count += 1
                successful_charts += 1
            
            total_charts += 1
        
        if chart_count == 0:
            print("⚠️  No renderable charts found")
        else:
            print(f"✅ Successfully rendered {chart_count} chart(s)")
    
    print(f"\n" + "=" * 70)
    print(f"📊 CHART RENDERING SUMMARY:")
    print(f"  • Total Charts Found: {total_charts}")
    print(f"  • Successfully Rendered: {successful_charts}")
    print(f"  • Failed/Skipped: {total_charts - successful_charts}")
    print("=" * 70)
    
    return total_charts, successful_charts

# Render all charts from transformation data
print("🎨 Loading and rendering charts from transformation plotly data files...")
total_found, total_rendered = render_all_charts()


🎨 Loading and rendering charts from transformation plotly data files...

📊 RENDERING ALL TRANSFORMATION CHARTS

🧹 Duplicate Removal Charts
--------------------------------------------------

📈 Bar Chart



📈 Pie Chart


✅ Successfully rendered 2 chart(s)

🎯 Outlier Detection Charts
--------------------------------------------------

📈 Comparison Chart



📈 Pie Chart



📈 Outliers By Column


✅ Successfully rendered 3 chart(s)

🧽 Null Value Cleaning Charts
--------------------------------------------------

📈 Comparison Chart



📈 Missing By Column



📈 Pie Chart


✅ Successfully rendered 3 chart(s)

📊 CHART RENDERING SUMMARY:
  • Total Charts Found: 8
  • Successfully Rendered: 8
  • Failed/Skipped: 0


In [8]:
# =============================================================================
# SECTION 6: COMPREHENSIVE COMPARISON TABLES
# =============================================================================

def create_detailed_comparison_tables():
    """Create comprehensive comparison tables"""
    
    if not available_transformations:
        print("❌ No transformation data available for comparison tables")
        return
    
    print(f"\n{'='*80}")
    print("📋 COMPREHENSIVE COMPARISON TABLES")
    print(f"{'='*80}")
    
    # =========================================================================
    # Table 1: Overview Comparison
    # =========================================================================
    
    print(f"\n📊 Table 1: Transformation Overview Comparison")
    print("-" * 80)
    
    overview_data = []
    
    # Raw data baseline
    overview_data.append({
        'Stage': '🗃️ Raw Data',
        'Rows': f"{raw_data_shape[0]:,}",
        'Columns': f"{raw_data_shape[1]:,}",
        'Data Issues': 'Not analyzed',
        'Processing Time': '-',
        'Quality Impact': 'Baseline'
    })
    
    # Add each transformation
    for trans_name in ['deduplication', 'outlier_removal', 'null_cleaning']:
        if trans_name in available_transformations:
            trans_data = available_transformations[trans_name]
            stats = trans_data['stats']
            config = trans_data['config']
            
            # Calculate processing time if available
            processing_time = 'Not recorded'
            if 'timestamp' in stats:
                processing_time = 'Completed'
            
            # Determine data issues found
            if trans_name == 'deduplication':
                issues = f"{stats.get('initial_duplicates', 0):,} duplicates"
            elif trans_name == 'outlier_removal':
                issues = f"{stats.get('outliers_detected', 0):,} outliers"
            elif trans_name == 'null_cleaning':
                issues = f"{stats.get('initial_nulls', 0):,} null values"
            
            # Calculate quality impact
            initial_rows = stats.get('initial_rows', raw_data_shape[0])
            final_rows = stats.get('final_rows', initial_rows)
            impact = f"{((final_rows / initial_rows) * 100):.1f}% data retained" if initial_rows > 0 else "No change"
            
            overview_data.append({
                'Stage': config['title'],
                'Rows': f"{final_rows:,}",
                'Columns': f"{stats.get('final_columns', raw_data_shape[1]):,}",
                'Data Issues': issues,
                'Processing Time': processing_time,
                'Quality Impact': impact
            })
    
    overview_df = pd.DataFrame(overview_data)
    print(overview_df.to_string(index=False, max_colwidth=20))
    
    # =========================================================================
    # Table 2: Detailed Metrics Comparison
    # =========================================================================
    
    print(f"\n\n📊 Table 2: Detailed Metrics Comparison")
    print("-" * 80)
    
    metrics_data = []
    
    for trans_name, trans_data in available_transformations.items():
        stats = trans_data['stats']
        config = trans_data['config']
        
        if trans_name == 'deduplication':
            metrics_data.append({
                'Transformation': config['title'],
                'Method/Strategy': f"Keep {stats.get('keep', 'first')}",
                'Items Detected': f"{stats.get('initial_duplicates', 0):,}",
                'Items Processed': f"{stats.get('removed_rows', 0):,}",
                'Success Rate': f"{stats.get('removal_percentage', 0):.2f}%",
                'Data Reduction': f"{stats.get('removed_rows', 0):,} rows"
            })
            
        elif trans_name == 'outlier_removal':
            method = stats.get('method', 'unknown').upper()
            threshold = stats.get('threshold', 'default')
            action = stats.get('action_details', {}).get('action', 'unknown')
            
            metrics_data.append({
                'Transformation': config['title'],
                'Method/Strategy': f"{method} (threshold: {threshold})",
                'Items Detected': f"{stats.get('outliers_detected', 0):,}",
                'Items Processed': f"{stats.get('rows_removed', 0):,}",
                'Success Rate': f"{stats.get('outlier_percentage', 0):.2f}%",
                'Data Reduction': f"{stats.get('rows_removed', 0):,} rows ({action})"
            })
            
        elif trans_name == 'null_cleaning':
            strategy = stats.get('strategy', 'unknown').upper()
            
            metrics_data.append({
                'Transformation': config['title'],
                'Method/Strategy': strategy,
                'Items Detected': f"{stats.get('initial_nulls', 0):,}",
                'Items Processed': f"{stats.get('nulls_removed', 0):,}",
                'Success Rate': f"{((stats.get('nulls_removed', 0) / max(stats.get('initial_nulls', 1), 1)) * 100):.2f}%",
                'Data Reduction': f"{stats.get('rows_removed', 0):,} rows + {stats.get('columns_removed', 0):,} cols"
            })
    
    if metrics_data:
        metrics_df = pd.DataFrame(metrics_data)
        print(metrics_df.to_string(index=False, max_colwidth=25))
    else:
        print("No detailed metrics available")
    
    # =========================================================================
    # Table 3: Column-wise Impact Analysis (if available)
    # =========================================================================
    
    print(f"\n\n📊 Table 3: Column-wise Impact Analysis")
    print("-" * 80)
    
    column_impact_data = []
    
    # Check for column-wise outlier data
    if 'outlier_removal' in available_transformations:
        outlier_stats = available_transformations['outlier_removal']['stats']
        if 'outlier_details' in outlier_stats:
            outlier_details = outlier_stats['outlier_details']
            
            for col, details in outlier_details.items():
                if col != 'isolation_forest':  # Skip method summary
                    column_impact_data.append({
                        'Column': col,
                        'Transformation': '🎯 Outlier Detection',
                        'Issues Found': f"{details.get('outlier_count', 0):,}",
                        'Impact Rate': f"{details.get('outlier_percentage', 0):.2f}%",
                        'Method Details': details.get('method', 'Unknown')
                    })
    
    # Check for column-wise null data
    if 'null_cleaning' in available_transformations:
        null_stats = available_transformations['null_cleaning']['stats']
        if 'missing_before' in null_stats:
            missing_before = null_stats['missing_before']
            missing_after = null_stats.get('missing_after', {})
            
            # Top 10 columns with missing values
            sorted_cols = sorted(missing_before.items(), 
                               key=lambda x: x[1]['missing_percentage'], 
                               reverse=True)[:10]
            
            for col, before_info in sorted_cols:
                before_pct = before_info['missing_percentage']
                after_pct = missing_after.get(col, {}).get('missing_percentage', 0)
                improvement = before_pct - after_pct
                
                column_impact_data.append({
                    'Column': col,
                    'Transformation': '🧽 Null Cleaning',
                    'Issues Found': f"{before_info.get('missing_count', 0):,}",
                    'Impact Rate': f"{before_pct:.1f}% → {after_pct:.1f}%",
                    'Method Details': f"Improved by {improvement:.1f}%"
                })
    
    if column_impact_data:
        column_df = pd.DataFrame(column_impact_data)
        print(column_df.to_string(index=False, max_colwidth=25))
    else:
        print("No column-wise impact data available")
    
    # =========================================================================
    # Table 4: Before/After Summary Statistics
    # =========================================================================
    
    print(f"\n\n📊 Table 4: Before/After Summary Statistics")
    print("-" * 80)
    
    summary_stats = {
        'Metric': [
            'Total Rows',
            'Total Columns', 
            'Duplicate Rows',
            'Outlier Rows',
            'Null Values',
            'Data Quality Score'
        ],
        'Before Processing': [],
        'After Processing': [],
        'Change': [],
        'Improvement %': []
    }
    
    # Calculate before/after values
    initial_rows = raw_data_shape[0]
    final_rows = initial_rows
    initial_cols = raw_data_shape[1]
    final_cols = initial_cols
    
    initial_duplicates = 0
    final_duplicates = 0
    initial_outliers = 0
    final_outliers = 0
    initial_nulls = 0
    final_nulls = 0
    
    # Aggregate data from transformations
    for trans_name, trans_data in available_transformations.items():
        stats = trans_data['stats']
        
        if trans_name == 'deduplication':
            initial_duplicates = stats.get('initial_duplicates', 0)
            final_duplicates = 0  # Assuming all duplicates removed
            final_rows = stats.get('final_rows', final_rows)
            
        elif trans_name == 'outlier_removal':
            initial_outliers = stats.get('outliers_detected', 0)
            final_outliers = 0 if stats.get('action_details', {}).get('action') == 'remove' else initial_outliers
            final_rows = stats.get('final_rows', final_rows)
            
        elif trans_name == 'null_cleaning':
            initial_nulls = stats.get('initial_nulls', 0)
            final_nulls = stats.get('final_nulls', 0)
            final_rows = stats.get('final_rows', final_rows)
            final_cols = stats.get('final_columns', final_cols)
    
    # Calculate quality scores (simplified)
    initial_quality = max(0, 100 - (initial_duplicates + initial_outliers + initial_nulls) / max(initial_rows * initial_cols, 1) * 100)
    final_quality = max(0, 100 - (final_duplicates + final_outliers + final_nulls) / max(final_rows * final_cols, 1) * 100)
    
    # Populate summary statistics
    before_values = [initial_rows, initial_cols, initial_duplicates, initial_outliers, initial_nulls, initial_quality]
    after_values = [final_rows, final_cols, final_duplicates, final_outliers, final_nulls, final_quality]
    
    for before, after in zip(before_values, after_values):
        change = after - before
        improvement = ((after - before) / max(before, 1) * 100) if before != 0 else 0
        
        summary_stats['Before Processing'].append(f"{before:,.1f}" if isinstance(before, float) else f"{before:,}")
        summary_stats['After Processing'].append(f"{after:,.1f}" if isinstance(after, float) else f"{after:,}")
        summary_stats['Change'].append(f"{change:+,.1f}" if isinstance(change, float) else f"{change:+,}")
        summary_stats['Improvement %'].append(f"{improvement:+.1f}%")
    
    summary_df = pd.DataFrame(summary_stats)
    print(summary_df.to_string(index=False))
    
    # =========================================================================
    # Table 5: Transformation Effectiveness Ranking
    # =========================================================================
    
    print(f"\n\n📊 Table 5: Transformation Effectiveness Ranking")
    print("-" * 80)
    
    effectiveness_data = []
    
    for trans_name, trans_data in available_transformations.items():
        stats = trans_data['stats']
        config = trans_data['config']
        
        # Calculate effectiveness score
        if trans_name == 'deduplication':
            issues_found = stats.get('initial_duplicates', 0)
            issues_resolved = stats.get('removed_rows', 0)
            data_retained = stats.get('final_rows', 1) / max(stats.get('initial_rows', 1), 1)
            
        elif trans_name == 'outlier_removal':
            issues_found = stats.get('outliers_detected', 0)
            issues_resolved = stats.get('rows_removed', 0)
            data_retained = stats.get('final_rows', 1) / max(stats.get('initial_rows', 1), 1)
            
        elif trans_name == 'null_cleaning':
            issues_found = stats.get('initial_nulls', 0)
            issues_resolved = stats.get('nulls_removed', 0)
            data_retained = stats.get('final_rows', 1) / max(stats.get('initial_rows', 1), 1)
        
        # Calculate effectiveness score (0-100)
        resolution_rate = (issues_resolved / max(issues_found, 1)) * 100 if issues_found > 0 else 100
        retention_score = data_retained * 100
        effectiveness_score = (resolution_rate * 0.7) + (retention_score * 0.3)  # Weight resolution higher
        
        effectiveness_data.append({
            'Transformation': config['title'],
            'Issues Found': f"{issues_found:,}",
            'Issues Resolved': f"{issues_resolved:,}",
            'Resolution Rate': f"{resolution_rate:.1f}%",
            'Data Retention': f"{retention_score:.1f}%",
            'Effectiveness Score': f"{effectiveness_score:.1f}",
            'Rank': 0  # Will be filled after sorting
        })
    
    # Sort by effectiveness score and assign ranks
    effectiveness_data.sort(key=lambda x: float(x['Effectiveness Score']), reverse=True)
    for i, item in enumerate(effectiveness_data, 1):
        item['Rank'] = i
        
        # Add emoji for ranking
        if i == 1:
            item['Rank'] = f"🥇 {i}"
        elif i == 2:
            item['Rank'] = f"🥈 {i}"
        elif i == 3:
            item['Rank'] = f"🥉 {i}"
        else:
            item['Rank'] = f"#{i}"
    
    if effectiveness_data:
        effectiveness_df = pd.DataFrame(effectiveness_data)
        print(effectiveness_df.to_string(index=False, max_colwidth=20))
    else:
        print("No effectiveness data available")
    
    print(f"\n{'='*80}")
    print("✅ All comparison tables generated successfully!")
    print(f"{'='*80}")

# Generate all comparison tables
create_detailed_comparison_tables()



📋 COMPREHENSIVE COMPARISON TABLES

📊 Table 1: Transformation Overview Comparison
--------------------------------------------------------------------------------
               Stage  Rows Columns     Data Issues Processing Time       Quality Impact
         🗃️ Raw Data 2,918      42    Not analyzed               -             Baseline
 🧹 Duplicate Removal   200      42    0 duplicates       Completed 100.0% data retained
 🎯 Outlier Detection   200      42  2,718 outliers       Completed   6.9% data retained
🧽 Null Value Clea...   200      42 224 null values       Completed 100.0% data retained


📊 Table 2: Detailed Metrics Comparison
--------------------------------------------------------------------------------
       Transformation      Method/Strategy Items Detected Items Processed Success Rate      Data Reduction
  🧹 Duplicate Removal           Keep first              0               0        0.00%              0 rows
  🎯 Outlier Detection IQR (threshold: 1.5)          2,718    

In [9]:
# =============================================================================
# SECTION 7: RECOMMENDATIONS AND NEXT STEPS
# =============================================================================

def generate_recommendations():
    """Generate recommendations based on transformation results"""
    
    if not available_transformations:
        return
    
    print(f"\n{'='*60}")
    print("💡 RECOMMENDATIONS & NEXT STEPS")
    print(f"{'='*60}")
    
    recommendations = []
    
    # Analyze each transformation
    for trans_name, trans_data in available_transformations.items():
        stats = trans_data['stats']
        config = trans_data['config']
        
        if trans_name == 'deduplication':
            removal_rate = stats.get('removal_percentage', 0)
            if removal_rate > 10:
                recommendations.append(
                    f"🔍 High duplicate rate ({removal_rate:.1f}%) detected. "
                    f"Consider investigating data collection processes."
                )
            elif removal_rate > 0:
                recommendations.append(
                    f"✅ Duplicate removal successful ({removal_rate:.1f}% removed)."
                )
            else:
                recommendations.append("✅ No duplicates found - excellent data quality!")
        
        elif trans_name == 'outlier_removal':
            outlier_rate = stats.get('outlier_percentage', 0)
            method = stats.get('method', 'unknown')
            
            if outlier_rate > 5:
                recommendations.append(
                    f"⚠️ High outlier rate ({outlier_rate:.1f}%) with {method} method. "
                    f"Consider trying different detection methods or reviewing data sources."
                )
            elif outlier_rate > 0:
                recommendations.append(
                    f"🎯 Outlier detection completed ({outlier_rate:.1f}% detected using {method})."
                )
            else:
                recommendations.append(f"✅ No outliers detected using {method} method.")
        
        elif trans_name == 'null_cleaning':
            nulls_removed = stats.get('nulls_removed', 0)
            final_nulls = stats.get('final_nulls', 0)
            strategy = stats.get('strategy', 'unknown')
            
            if final_nulls > 0:
                recommendations.append(
                    f"🧹 Null cleaning with {strategy} strategy: {nulls_removed:,} nulls cleaned, "
                    f"{final_nulls:,} remaining. Consider additional cleaning strategies if needed."
                )
            else:
                recommendations.append(f"✅ All null values successfully cleaned using {strategy} strategy!")
    
    # General recommendations
    if len(available_transformations) < 3:
        missing_transforms = []
        if 'deduplication' not in available_transformations:
            missing_transforms.append("duplicate removal")
        if 'outlier_removal' not in available_transformations:
            missing_transforms.append("outlier detection")
        if 'null_cleaning' not in available_transformations:
            missing_transforms.append("null value cleaning")
        
        if missing_transforms:
            recommendations.append(
                f"🔄 Consider running additional transformations: {', '.join(missing_transforms)}"
            )
    
    # Print recommendations
    for i, rec in enumerate(recommendations, 1):
        print(f"{i}. {rec}")
    
    # Next steps
    print(f"\n🚀 Next Steps:")
    print("1. Review the visualizations and statistics above")
    print("2. If quality scores are low, consider adjusting transformation parameters") 
    print("3. Run additional transformations if needed")
    print("4. Export cleaned data for your machine learning or analysis pipeline")
    print("5. Consider setting up automated pipeline monitoring")
    
    # Data export suggestions
    print(f"\n📁 Data Export Options:")
    for trans_name, trans_data in available_transformations.items():
        config = trans_data['config']
        data_path = os.path.join(trans_data['folder_path'], config['data_file'])
        if os.path.exists(data_path):
            print(f"  • {config['title']}: {data_path}")

# Generate recommendations
generate_recommendations()

print(f"\n{'='*60}")
print("🎉 VISUALIZATION DASHBOARD COMPLETE!")
print(f"{'='*60}")
print("📊 All available transformation results have been analyzed and visualized.")
print("💡 Review the recommendations above to further improve your data quality.")
print("🔄 Re-run this notebook anytime after performing new transformations.")
print(f"{'='*60}")


💡 RECOMMENDATIONS & NEXT STEPS
1. ✅ No duplicates found - excellent data quality!
2. ⚠️ High outlier rate (93.1%) with iqr method. Consider trying different detection methods or reviewing data sources.
3. 🧹 Null cleaning with fill_median strategy: 0 nulls cleaned, 224 remaining. Consider additional cleaning strategies if needed.

🚀 Next Steps:
1. Review the visualizations and statistics above
2. If quality scores are low, consider adjusting transformation parameters
3. Run additional transformations if needed
4. Export cleaned data for your machine learning or analysis pipeline
5. Consider setting up automated pipeline monitoring

📁 Data Export Options:
  • 🧹 Duplicate Removal: ./pipeline_output/deduplicated/deduplicated_data.pkl
  • 🎯 Outlier Detection: ./pipeline_output/outliers_removed/outliers_removed_data.pkl
  • 🧽 Null Value Cleaning: ./pipeline_output/null_cleaned/nulls_cleaned_data.pkl

🎉 VISUALIZATION DASHBOARD COMPLETE!
📊 All available transformation results have been analyz